# Module 10.3: Continuous Batching & PagedAttention

In Notebook 09, we implemented standard **KV Caching**. It allows us to avoid re-computing Attention values for tokens we've already processed. 

However, when serving an LLM to thousands of users simultaneously (like OpenAI or Anthropic does), standard KV Caching creates a massive problem: **Memory Fragmentation**.

## 1. The Fragmentation Problem

In standard batching, you must pre-allocate contiguous chunks of GPU memory for each user's KV Cache up to their *maximum* sequence length. 

If User A generates 100 tokens, and User B generates 2000 tokens, the GPU memory allocator has to guess the max length. Pre-allocating wastes huge amounts of memory if the output is short. But dynamic allocation leads to fragmentation: tiny holes of free memory scattered across the GPU that are too small to fit a new user's contiguous cache block.

**Result**: Only 20-40% of the GPU VRAM is actually utilized for caching. The rest is wasted.

## 2. The Solution: PagedAttention (vLLM)

### Borrowing from Operating Systems
Decades ago, CPU RAM solved fragmentation using **Virtual Memory and Paging**. 
Instead of finding a contiguous megabyte for a program, the OS splits RAM into strict 4KB "Pages". An application gets a Virtual memory map that points to scattered Physical Pages. 

**PagedAttention** does this for GPU KV Caches:
1. We slice the KV Cache into fixed-size "Blocks" (e.g., each block holds 16 tokens).
2. We maintain a **Block Table** map.
3. When computing Attention, the kernel fetches the blocks piece-by-piece from scattered physical memory locations, instead of requiring one massive contiguous tensor.

In [1]:
import torch

# Constants
BLOCK_SIZE = 4            # Tokens per block (normally 16 or 32)
NUM_BLOCKS = 10           # Total physical blocks available in VRAM
HEAD_DIM = 64             # Size of K/V vectors per token

# 1. The Physical KV Cache Pool (Scattered in VRAM)
# Shape: [NUM_BLOCKS, BLOCK_SIZE, HEAD_DIM]
physical_k_cache = torch.zeros(NUM_BLOCKS, BLOCK_SIZE, HEAD_DIM)

# 2. The Block Table (The Virtual-to-Physical Lookup) for a single User Request
# -1 means unallocated.
block_table = [-1, -1, -1, -1] 

print(f"Physical KV Cache Shape: {physical_k_cache.shape}")

## 3. Allocating and Filling Blocks

Let's simulate processing a request of 6 tokens.

In [2]:
seq_len = 6

# We need enough physical blocks to hold 6 tokens. (6 / 4 = 1 remainder 2, so 2 blocks needed)
blocks_needed = (seq_len + BLOCK_SIZE - 1) // BLOCK_SIZE

# Let's artificially simulate that memory is fragmented.
# Block 0, 1, 2, 4 are "in use" by other people.
available_physical_blocks = [3, 5, 6, 7, 8, 9]

# Allocate!
print(f"Sequence Length: {seq_len} -> Requires {blocks_needed} Blocks.")
for i in range(blocks_needed):
    physical_block_idx = available_physical_blocks.pop(0)
    block_table[i] = physical_block_idx

print(f"Virtual Block Table Map: {block_table}")

# Inject the 6 tokens into the physical scattered cache
mock_k_tensors = torch.randn(6, HEAD_DIM)

for i in range(seq_len):
    logical_block_idx = i // BLOCK_SIZE
    offset_in_block = i % BLOCK_SIZE
    
    physical_block_idx = block_table[logical_block_idx]
    
    # Write to scattered memory!
    physical_k_cache[physical_block_idx, offset_in_block] = mock_k_tensors[i]
    
print("Tokens successfully injected into scattered physical memory pages.")

## 4. Continuous Batching

Because PagedAttention completely decouples memory allocation from sequence length, we can do **Continuous Batching**.

In older systems, you wait for 4 users to submit requests. You batch them. If User A finishes after 10 tokens, and User D takes 2000 tokens, the GPU slot for User A stays "locked" and idle while waiting for D to finish. 

With Continuous Batching, the moment User A's generation finishes, their Blocks are returned to the pool, and the server immediately injects User E into the batch at iteration 11. The batch size is dynamic and iterating at the token level, resulting in up to **40x higher serving throughput**!